In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import statsmodels.api as sm

from src.data import *
from src.models import *
from src.features import log_returns, realised_volatility
from src.backtest import walk_forward
from src.metrics import compare_models, dm_table

/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
spot = load_brent_spot(start="1987-01-01")
data = spot.copy()
data["log_return"] = log_returns(data, col="brent_spot")
data = data.dropna()


test_args = dict(horizon=60, start_date="2024-09-30", end_date="2026-09-30", target_end_date="2026-09-30")


test_results = {
    "Random Walk": walk_forward(data, forecast_funct=random_walk_forecast, **test_args),
    "Historical Mean": walk_forward(data, forecast_funct=historical_mean_forecast, lookback=3000, **test_args),
    "AR": walk_forward(data, forecast_funct=ar1_forecast, lookback=3000, **test_args),
    "Direct forecast": walk_forward(data, forecast_funct=direct_forecast, lookback=3000, **test_args),
    "Direct lag_1 lag_60": walk_forward(data, forecast_funct=direct_lags_forecast, short=1, long=60, lookback=3000, **test_args),
    "Direct lag_5 lag_60": walk_forward(data, forecast_funct=direct_lags_forecast, short=5, long=60, lookback=3000, **test_args)
}

compare_models(test_results, horizon=60)

,MAE,RMSE,MAPE,directional_accuracy,precision_up,recall_up,f1_up,precision_down,recall_down,f1_down,annualized_return,sharpe_ratio,max_drawdown,num_trades
model,,,,,,,,,,,,,,
Random Walk,14.490716,22.169808,16.181882,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.0
Historical Mean,14.576918,22.379678,16.223646,0.447427,0.000000,0.000000,0.000000,0.485437,0.851064,0.618238,-0.166297,-0.572241,-0.424695,1.0
AR,14.576318,22.376700,16.222905,0.445190,0.000000,0.000000,0.000000,0.484185,0.846809,0.616099,-0.166297,-0.572241,-0.424695,1.0
Direct forecast,12.246353,19.712169,13.330732,0.740492,0.763736,0.655660,0.705584,0.724528,0.817021,0.768000,-0.218351,-0.776423,-0.451486,7.0
Direct lag_1 lag_60,12.156660,19.564622,13.258065,0.736018,0.758242,0.650943,0.700508,0.720755,0.812766,0.764000,0.094767,0.316611,-0.180435,7.0
Direct lag_5 lag_60,12.231381,19.588360,13.355331,0.727069,0.739362,0.655660,0.695000,0.718147,0.791489,0.753036,-0.218351,-0.776423,-0.451486,7.0


### Mean Revert

In [3]:
# PRICES = ["brent_spot", "wti", "usd", "sp500"]   # prices -> log returns
# INDICES = ["vix", "ovx", "epu", "gpr"]           # fear/uncertainty indices -> log levels

df_mr = load_dataset()
df_mr = df_mr[["brent_spot", "wti"]]
df_mr = df_mr.rename(columns={"wti": "wti_spot"})
df_mr["log_return"] = np.log(df_mr["brent_spot"]).diff()
df_mr = df_mr.dropna()
df_mr.tail()

,brent_spot,wti_spot,log_return
DATE,,,
2026-10-21,117.37,103.95,0.018488
2026-10-22,125.39,104.27,0.066098
2026-10-23,128.30,109.72,0.022942
2026-10-26,128.12,105.25,-0.001404
2026-10-27,123.10,103.37,-0.039970


In [4]:
res = walk_forward(df_mr, forecast_funct=spread_mr_forecast, lookback=3000, **test_args)
res.tail()

,origin_date,target_date,origin_price,forecast,actual
442,2026-07-01,2026-09-24,69.24,74.467540,120.92
443,2026-07-02,2026-09-25,68.53,74.590084,116.01
444,2026-07-03,2026-09-28,68.68,74.634381,119.97
445,2026-07-06,2026-09-29,69.56,74.438928,113.96
446,2026-07-07,2026-09-30,71.78,76.512273,115.91


In [5]:
test_results["Mean Revert"] = res

compare_models(test_results, horizon=60)

,MAE,RMSE,MAPE,directional_accuracy,precision_up,recall_up,f1_up,precision_down,recall_down,f1_down,annualized_return,sharpe_ratio,max_drawdown,num_trades
model,,,,,,,,,,,,,,
Random Walk,14.490716,22.169808,16.181882,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.0
Historical Mean,14.576918,22.379678,16.223646,0.447427,0.000000,0.000000,0.000000,0.485437,0.851064,0.618238,-0.166297,-0.572241,-0.424695,1.0
AR,14.576318,22.376700,16.222905,0.445190,0.000000,0.000000,0.000000,0.484185,0.846809,0.616099,-0.166297,-0.572241,-0.424695,1.0
Direct forecast,12.246353,19.712169,13.330732,0.740492,0.763736,0.655660,0.705584,0.724528,0.817021,0.768000,-0.218351,-0.776423,-0.451486,7.0
Direct lag_1 lag_60,12.156660,19.564622,13.258065,0.736018,0.758242,0.650943,0.700508,0.720755,0.812766,0.764000,0.094767,0.316611,-0.180435,7.0
Direct lag_5 lag_60,12.231381,19.588360,13.355331,0.727069,0.739362,0.655660,0.695000,0.718147,0.791489,0.753036,-0.218351,-0.776423,-0.451486,7.0
Mean Revert,14.133748,21.567704,15.650951,0.425056,0.395349,0.400943,0.398126,0.452586,0.446809,0.449679,-0.371631,-1.592582,-0.552036,9.0


### XGBoost

In [6]:
res = walk_forward(data, forecast_funct=xgb_direct_forecast, lookback=3000, **test_args)
res.tail()

,origin_date,target_date,origin_price,forecast,actual
442,2026-07-01,2026-09-24,69.24,86.452067,120.92
443,2026-07-02,2026-09-25,68.53,89.377221,116.01
444,2026-07-03,2026-09-28,68.68,86.222259,119.97
445,2026-07-06,2026-09-29,69.56,87.064515,113.96
446,2026-07-07,2026-09-30,71.78,88.852946,115.91


In [7]:
test_results["XGBoost"] = res

compare_models(test_results, horizon=60)

,MAE,RMSE,MAPE,directional_accuracy,precision_up,recall_up,f1_up,precision_down,recall_down,f1_down,annualized_return,sharpe_ratio,max_drawdown,num_trades
model,,,,,,,,,,,,,,
Random Walk,14.490716,22.169808,16.181882,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.0
Historical Mean,14.576918,22.379678,16.223646,0.447427,0.000000,0.000000,0.000000,0.485437,0.851064,0.618238,-0.166297,-0.572241,-0.424695,1.0
AR,14.576318,22.376700,16.222905,0.445190,0.000000,0.000000,0.000000,0.484185,0.846809,0.616099,-0.166297,-0.572241,-0.424695,1.0
Direct forecast,12.246353,19.712169,13.330732,0.740492,0.763736,0.655660,0.705584,0.724528,0.817021,0.768000,-0.218351,-0.776423,-0.451486,7.0
Direct lag_1 lag_60,12.156660,19.564622,13.258065,0.736018,0.758242,0.650943,0.700508,0.720755,0.812766,0.764000,0.094767,0.316611,-0.180435,7.0
Direct lag_5 lag_60,12.231381,19.588360,13.355331,0.727069,0.739362,0.655660,0.695000,0.718147,0.791489,0.753036,-0.218351,-0.776423,-0.451486,7.0
Mean Revert,14.133748,21.567704,15.650951,0.425056,0.395349,0.400943,0.398126,0.452586,0.446809,0.449679,-0.371631,-1.592582,-0.552036,9.0
XGBoost,16.214299,24.678568,18.584187,0.534676,0.506135,0.778302,0.613383,0.611570,0.314894,0.415730,0.085776,0.285877,-0.246929,7.0


### Direct Ridge Forecast

In [8]:
df_rr = load_dataset()
  
df_rr = df_rr[["brent_spot", "wti", "usd", "sp500", "vix", "ovx"]]
df_rr = df_rr.rename(columns={"wti": "wti_spot"})
df_rr = df_rr.dropna()

for col in ["usd", "sp500", "vix", "ovx"]:
    df_rr[col] = np.log(df_rr[col])

df_rr.tail()

,brent_spot,wti_spot,usd,sp500,vix,ovx
DATE,,,,,,
2026-10-21,117.37,103.95,4.798615,8.956373,2.814210,3.975561
2026-10-22,125.39,104.27,4.798704,8.944215,2.946017,4.018902
2026-10-23,128.30,109.72,4.798433,8.939014,2.967847,4.094678
2026-10-26,128.12,105.25,4.798433,8.945954,2.911807,4.072610
2026-10-27,123.10,103.37,4.798433,8.951906,2.863343,4.015301


In [9]:
res = walk_forward(df_rr, forecast_funct=direct_ridge_forecast, lookback=3000, **test_args)
res.tail()

,origin_date,target_date,origin_price,forecast,actual
442,2026-07-01,2026-09-24,69.24,69.183614,120.92
443,2026-07-02,2026-09-25,68.53,68.512663,116.01
444,2026-07-03,2026-09-28,68.68,68.782695,119.97
445,2026-07-06,2026-09-29,69.56,69.622541,113.96
446,2026-07-07,2026-09-30,71.78,71.915229,115.91


In [10]:
test_results["Direct ridge"] = res

compare_models(test_results,  horizon=60)

,MAE,RMSE,MAPE,directional_accuracy,precision_up,recall_up,f1_up,precision_down,recall_down,f1_down,annualized_return,sharpe_ratio,max_drawdown,num_trades
model,,,,,,,,,,,,,,
Random Walk,14.490716,22.169808,16.181882,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.0
Historical Mean,14.576918,22.379678,16.223646,0.447427,0.000000,0.000000,0.000000,0.485437,0.851064,0.618238,-0.166297,-0.572241,-0.424695,1.0
AR,14.576318,22.376700,16.222905,0.445190,0.000000,0.000000,0.000000,0.484185,0.846809,0.616099,-0.166297,-0.572241,-0.424695,1.0
Direct forecast,12.246353,19.712169,13.330732,0.740492,0.763736,0.655660,0.705584,0.724528,0.817021,0.768000,-0.218351,-0.776423,-0.451486,7.0
Direct lag_1 lag_60,12.156660,19.564622,13.258065,0.736018,0.758242,0.650943,0.700508,0.720755,0.812766,0.764000,0.094767,0.316611,-0.180435,7.0
Direct lag_5 lag_60,12.231381,19.588360,13.355331,0.727069,0.739362,0.655660,0.695000,0.718147,0.791489,0.753036,-0.218351,-0.776423,-0.451486,7.0
Mean Revert,14.133748,21.567704,15.650951,0.425056,0.395349,0.400943,0.398126,0.452586,0.446809,0.449679,-0.371631,-1.592582,-0.552036,9.0
XGBoost,16.214299,24.678568,18.584187,0.534676,0.506135,0.778302,0.613383,0.611570,0.314894,0.415730,0.085776,0.285877,-0.246929,7.0
Direct ridge,14.265363,22.079316,15.857825,0.689038,0.665158,0.693396,0.678984,0.712389,0.685106,0.698482,0.266544,0.989700,-0.142001,11.0


### Combination Forecast

In [11]:
from functools import partial
from scipy import stats

def combination_forecast(df, origin_idx, horizon=60, fit_idx=None, members=None, weights=None):
    """Weighted average of member forecasts.
    Members are existing forecast functions with signature (df, origin_idx, horizon, fit_idx).
    Averaging is done on the predicted 60-day LOG return, then converted back to a price,
    so the combination is a geometric mean of the member price forecasts."""
    fit_idx = origin_idx if fit_idx is None else fit_idx
    assert fit_idx <= origin_idx, "fit_idx must not be after origin_idx (lookahead)"

    if members is None:   # choose members BEFORE looking at results; keep RW in as a shrinkage anchor
        members = [
            random_walk_forecast,
            partial(direct_lags_forecast, short=1, long=60, lookback=3000),
            partial(xgb_direct_forecast, lookback=3000),
            partial(direct_forecast, lookback=3000),
        ]
        
    w = np.full(len(members), 1 / len(members)) if weights is None else np.asarray(weights, float)
    assert len(w) == len(members) and np.isclose(w.sum(), 1), "weights must match members and sum to 1"

    p_t = df["brent_spot"].iloc[origin_idx]
    rets = np.array([np.log(fn(df, origin_idx, horizon=horizon, fit_idx=fit_idx) / p_t)
                     for fn in members])
    return p_t * np.exp(w @ rets)

In [12]:
weights = [0.1, 0.35, 0.2, 0.35]
res = walk_forward(data, forecast_funct=combination_forecast, weights=weights, **test_args)
res.tail()

,origin_date,target_date,origin_price,forecast,actual
442,2026-07-01,2026-09-24,69.24,82.343744,120.92
443,2026-07-02,2026-09-25,68.53,83.870617,116.01
444,2026-07-03,2026-09-28,68.68,81.376332,119.97
445,2026-07-06,2026-09-29,69.56,81.808305,113.96
446,2026-07-07,2026-09-30,71.78,83.805027,115.91


In [13]:
test_results["Combination"] = res

compare_models(test_results, horizon=60)

,MAE,RMSE,MAPE,directional_accuracy,precision_up,recall_up,f1_up,precision_down,recall_down,f1_down,annualized_return,sharpe_ratio,max_drawdown,num_trades
model,,,,,,,,,,,,,,
Random Walk,14.490716,22.169808,16.181882,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.0
Historical Mean,14.576918,22.379678,16.223646,0.447427,0.000000,0.000000,0.000000,0.485437,0.851064,0.618238,-0.166297,-0.572241,-0.424695,1.0
AR,14.576318,22.376700,16.222905,0.445190,0.000000,0.000000,0.000000,0.484185,0.846809,0.616099,-0.166297,-0.572241,-0.424695,1.0
Direct forecast,12.246353,19.712169,13.330732,0.740492,0.763736,0.655660,0.705584,0.724528,0.817021,0.768000,-0.218351,-0.776423,-0.451486,7.0
Direct lag_1 lag_60,12.156660,19.564622,13.258065,0.736018,0.758242,0.650943,0.700508,0.720755,0.812766,0.764000,0.094767,0.316611,-0.180435,7.0
Direct lag_5 lag_60,12.231381,19.588360,13.355331,0.727069,0.739362,0.655660,0.695000,0.718147,0.791489,0.753036,-0.218351,-0.776423,-0.451486,7.0
Mean Revert,14.133748,21.567704,15.650951,0.425056,0.395349,0.400943,0.398126,0.452586,0.446809,0.449679,-0.371631,-1.592582,-0.552036,9.0
XGBoost,16.214299,24.678568,18.584187,0.534676,0.506135,0.778302,0.613383,0.611570,0.314894,0.415730,0.085776,0.285877,-0.246929,7.0
Direct ridge,14.265363,22.079316,15.857825,0.689038,0.665158,0.693396,0.678984,0.712389,0.685106,0.698482,0.266544,0.989700,-0.142001,11.0


### ==== Diebold-Mariano test ====

In [14]:
long_args = dict(horizon=60, start_date="2024-09-30", end_date="2026-09-30", target_end_date="2026-09-30")

results = {
    "Random Walk":         walk_forward(data, forecast_funct=random_walk_forecast, **long_args),
    "Direct lag_1 lag_60": walk_forward(data, forecast_funct=direct_lags_forecast, short=1, long=60, lookback=3000, **long_args),
    "Direct forecast":     walk_forward(data, forecast_funct=direct_forecast, lookback=3000, **long_args),
    "Combination":         walk_forward(data, forecast_funct=combination_forecast, **long_args),
}

# compare_models(results, horizon=60)
for loss in ["ae", "se"]:
    print(f"\nDM test vs Random Walk, loss = {loss}")
    print(dm_table(results, benchmark="Random Walk", horizon=60, loss=loss).round(4))


DM test vs Random Walk, loss = ae
                     DM_stat  p_value      n
Direct lag_1 lag_60  -1.0670   0.2865  447.0
Direct forecast      -1.0726   0.2841  447.0
Combination          -1.1003   0.2718  447.0

DM test vs Random Walk, loss = se
                     DM_stat  p_value      n
Direct lag_1 lag_60  -1.0670   0.2865  447.0
Direct forecast      -1.0726   0.2841  447.0
Combination          -1.1003   0.2718  447.0


### ==== Final 60-day forecast =====
origin = 2026-09-30

In [15]:
ORIGIN_DATE = "2026-09-30"
HORIZON, LOOKBACK = 60, 3000        # 60 trading days ahead, 3000-day fitting window
SHORT, LONG = 1, 60                 # r_1d and lagged r_60d (non-overlapping, days t-59..t-1)

# 1) Locate the origin (last available trading day on or before ORIGIN_DATE)
dates = data.index
origin_idx = dates.searchsorted(pd.Timestamp(ORIGIN_DATE), side="right") - 1
origin_date = dates[origin_idx]
if origin_date != pd.Timestamp(ORIGIN_DATE):
    print(f"WARNING: {ORIGIN_DATE} not in data, using {origin_date.date()} instead")
fit_idx = origin_idx                # model fitted only on targets known by the origin
start = fit_idx - LOOKBACK + 1
assert start >= 0, "not enough history"

# 2) Rebuild the training set exactly as direct_lags_forecast does (so we can inspect the fit)
lp = np.log(data["brent_spot"].to_numpy(float))        # log of PRICE (not of log_return)

def make_X(t):
    return pd.DataFrame({
        "r_1d":         lp[t] - lp[t - SHORT],         # latest 1-day log return
        "r_60d_lagged": lp[t - SHORT] - lp[t - LONG],  # log return over t-59..t-1
    })

t = np.arange(start + LONG, fit_idx - HORIZON + 1)     # last target = fit_idx
X = make_X(t)
y = pd.Series(lp[t + HORIZON] - lp[t], name="ret_60d_ahead")

fit = sm.OLS(y.values, sm.add_constant(X)).fit(cov_type="HAC", cov_kwds={"maxlags": HORIZON})

# 3) Parameters and significance (Newey-West/HAC, because 60-day targets overlap)
print(fit.summary())
ci = np.asarray(fit.conf_int())
names = fit.model.exog_names
params = pd.DataFrame({"coef": np.asarray(fit.params), "HAC se": np.asarray(fit.bse),
                       "t": np.asarray(fit.tvalues), "p-value": np.asarray(fit.pvalues),
                       "95% low": ci[:, 0], "95% high": ci[:, 1]}, index=names)
print("\n", params.round(4))
print(f"\nobs = {len(y)}  (~{len(y)//HORIZON} non-overlapping 60-day windows)")
print(f"R2 = {fit.rsquared:.4f}   joint Wald p-value = {fit.f_pvalue:.4f}")

# 4) Forecast from the origin
x0 = np.r_[1.0, make_X(np.array([origin_idx])).to_numpy()[0]]
beta = np.asarray(fit.params)
pred_ret = float(x0 @ beta)
se_mean = float(np.sqrt(x0 @ np.asarray(fit.cov_params()) @ x0))   # uncertainty of expected return
sigma = float(np.sqrt(fit.scale))                       # residual sd of a 60-day return
sd_total = np.sqrt(se_mean**2 + sigma**2)
p0 = data["brent_spot"].iloc[origin_idx]
f = lambda r: p0 * np.exp(r)

# sanity check against the function used in the backtest
fc_fn = direct_lags_forecast(data, origin_idx, horizon=HORIZON, fit_idx=fit_idx,
                             lookback=LOOKBACK, short=SHORT, long=LONG,
                             nonoverlap=True, use_intercept=True)
assert np.isclose(f(pred_ret), fc_fn), "cell and model function disagree"

target_date = pd.bdate_range(origin_date, periods=HORIZON + 1)[-1]   # approx: ignores holidays
print(f"\nOrigin {origin_date.date()}  price = {p0:.2f}")
print(f"Features: r_1d = {x0[1]:+.4f}, r_60d_lagged = {x0[2]:+.4f}")
print(f"Predicted 60-day log return = {pred_ret:+.4f}  (expected-return 95% CI "
      f"{pred_ret-1.96*se_mean:+.4f} to {pred_ret+1.96*se_mean:+.4f})")
print(f"Point forecast for ~{target_date.date()}: {f(pred_ret):.2f}")
print(f"95% prediction interval: {f(pred_ret-1.96*sd_total):.2f} to {f(pred_ret+1.96*sd_total):.2f}")
print(f"Random-walk forecast: {p0:.2f}")

# 5) Optional: how the forecast is doing so far, if data already runs past the origin
elapsed = len(data) - 1 - origin_idx
if elapsed > 0:
    print(f"\n{elapsed} trading days elapsed. Latest {data.index[-1].date()}: "
          f"{data['brent_spot'].iloc[-1]:.2f} (realised log return {lp[-1]-lp[origin_idx]:+.4f})")

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.121
Model:                            OLS   Adj. R-squared:                  0.120
Method:                 Least Squares   F-statistic:                     7.035
Date:                Fri, 30 Oct 2026   Prob (F-statistic):           0.000896
Time:                        17:55:52   Log-Likelihood:                 166.33
No. Observations:                2880   AIC:                            -326.7
Df Residuals:                    2877   BIC:                            -308.8
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.0144      0.027      0.533   

#### GARCH-based prediction interval

In [16]:
from arch import arch_model

def garch_interval_forecast(df, origin_idx, horizon=60, fit_idx=None, lookback=3000,
                            center_fn=None, level=0.95, vol="garch", gjr=False):
    """Prediction interval for Brent at origin+horizon.
    - centre : log-return forecast of `center_fn` (default: random walk, 0 return)
    - width  : sqrt of the summed h-step GARCH(1,1)-t variance forecasts
               (vol="constant" gives the benchmark sd*sqrt(h) from the fit window)
    - params are estimated on data up to fit_idx only; the volatility state is then
      filtered up to origin_idx with params frozen (no look-ahead).
    Returns dict(forecast, lower, upper, sigma_h)."""
    fit_idx = origin_idx if fit_idx is None else fit_idx
    assert fit_idx <= origin_idx, "fit_idx must not be after origin_idx (lookahead)"
    start = fit_idx - lookback + 1
    assert start >= 0, "not enough history for the lookback window"

    r = df["log_return"].to_numpy(float)
    assert not np.isnan(r[start: origin_idx + 1]).any(), "NaNs in return window"
    p_t = df["brent_spot"].iloc[origin_idx]

    used = vol
    if vol == "garch":
        spec = dict(mean="Zero", vol="GARCH", p=1, o=int(gjr), q=1, dist="t", rescale=False)
        fit = arch_model(r[start: fit_idx + 1] * 100, **spec).fit(disp="off")   # pct scale
        if fit.convergence_flag != 0:
            used = "constant (garch failed)"
        else:
            filt = arch_model(r[start: origin_idx + 1] * 100, **spec).fix(fit.params)
            var_path = filt.forecast(horizon=horizon, reindex=False).variance.to_numpy()[-1]
            sigma_h = np.sqrt(var_path.sum()) / 100          # sd of the h-day cumulative log return
    if used != "garch":
        sigma_h = r[start: fit_idx + 1].std(ddof=1) * np.sqrt(horizon)

    pred_ret = 0.0 if center_fn is None else np.log(center_fn(df, origin_idx, horizon, fit_idx) / p_t)
    z = stats.norm.ppf(0.5 + level / 2)    # 60-day sum of returns is close to normal even if daily is fat-tailed
    return {"forecast": p_t * np.exp(pred_ret),
            "lower": p_t * np.exp(pred_ret - z * sigma_h),
            "upper": p_t * np.exp(pred_ret + z * sigma_h),
            "sigma_h": sigma_h, "vol_used": used}


def walk_forward_interval(df, horizon, start_date, end_date, interval_fn,
                          target_end_date=None, **kwargs):
    """Same loop as walk_forward but records lower/upper too (result also works with the metric functions)."""
    dates = df.index
    start_idx, end_idx = dates.searchsorted(start_date), dates.searchsorted(end_date)
    rows = []
    for i in range(start_idx, end_idx):
        j = i + horizon
        if j >= len(dates): break
        if target_end_date is not None and dates[j] > pd.Timestamp(target_end_date): break
        out = interval_fn(df, i, horizon, i, **kwargs)
        rows.append({"origin_date": dates[i], "target_date": dates[j],
                     "origin_price": df["brent_spot"].iloc[i], **out,
                     "actual": df["brent_spot"].iloc[j]})
    return pd.DataFrame(rows)


def interval_metrics(res, level=0.95):
    """Coverage should be close to `level`. Interval (Winkler) score: lower = better (rewards narrow AND correct)."""
    a = 1 - level
    inside = (res["actual"] >= res["lower"]) & (res["actual"] <= res["upper"])
    width = res["upper"] - res["lower"]
    score = width + (2 / a) * (res["lower"] - res["actual"]).clip(lower=0) \
                  + (2 / a) * (res["actual"] - res["upper"]).clip(lower=0)
    return {"coverage": inside.mean(), "target": level, "mean_width": width.mean(),
            "mean_rel_width": (width / res["origin_price"]).mean(), "interval_score": score.mean(),
            "n": len(res), "eff_n (n/h)": len(res) / 60}   # overlapping windows: effective n is ~n/h

In [17]:
from functools import partial

origin_idx = data.index.searchsorted(pd.Timestamp("2026-09-30"), side="right") - 1
point_model = partial(direct_lags_forecast, short=1, long=60, lookback=3000)

garch_interval_forecast(data, origin_idx, horizon=60, fit_idx=origin_idx,
                              lookback=3000, center_fn=point_model, level=0.95)

{'forecast': 99.44237634709832,
 'lower': 62.53797929642956,
 'upper': 158.1244921055918,
 'sigma_h': 0.23663919691476404,
 'vol_used': 'garch'}